In [0]:
from pyspark.sql import functions as F

catalog, schema = "workspace", "patient_notes"
silver_table = f"{catalog}.{schema}.silver_notes"
chunk_table  = f"{catalog}.{schema}.silver_chunks"

In [0]:
silver = spark.table(silver_table)

chunks = (silver
    .select("note_id", "patient_id", "note_date", "note_type",
            F.posexplode(F.split("note_text", "\n")).alias("pos", "line"))   # one row per line = per section
    .filter(F.length(F.trim("line")) > 0)
    .withColumn("section", F.lower(F.regexp_extract("line", r"^([^:]+):", 1)))
    .withColumn("chunk_id", F.concat_ws("-", "note_id", F.col("pos").cast("string")))
    .withColumn("chunk_text", F.concat(                                         # contextual header + section text
        F.lit("Patient "), F.col("patient_id"), F.lit(" · "),
        F.col("note_date").cast("string"), F.lit(" · "), F.col("note_type"), F.lit(" · "),
        F.trim("line")))
    .select("chunk_id", "note_id", "patient_id", "note_date", "note_type", "section", "chunk_text"))

display(chunks.groupBy("note_type", "section").count().orderBy("note_type", "section"))
display(chunks.filter("note_type = 'discharge summary'").limit(5))

In [0]:
(chunks.write.mode("overwrite")
       .option("delta.enableChangeDataFeed", "true")      # lets the index see new/changed chunks
       .saveAsTable(chunk_table))

spark.sql(f"ALTER TABLE {chunk_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("Chunks:", spark.table(chunk_table).count())

In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

client        = AISearchClient()
endpoint_name = "patient_notes_search"
client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

In [0]:
index_name = "workspace.patient_notes.silver_chunks_index"

client.create_delta_sync_index(
    endpoint_name=endpoint_name,
    source_table_name="workspace.patient_notes.silver_chunks",
    index_name=index_name,
    pipeline_type="TRIGGERED",                    # syncs when we ask, not continuously (cheaper)
    primary_key="chunk_id",
    embedding_source_column="chunk_text",         # Databricks embeds this column for us
    embedding_model_endpoint_name="databricks-qwen3-embedding-0-6b")

In [0]:
from databricks.ai_search.client import AISearchClient

client     = AISearchClient()
index_name = "workspace.patient_notes.silver_chunks_index"
index      = client.get_index(index_name=index_name)

res = index.similarity_search(
    query_text="breathing problems",
    columns=["chunk_id", "patient_id", "section", "chunk_text"],
    num_results=5)

for row in res["result"]["data_array"]:
    print(row)

In [0]:
res = index.similarity_search(
    query_text="surgery history",
    columns=["chunk_id", "patient_id", "section", "chunk_text"],
    filters={"patient_id": "P0022"},
    num_results=5)
for row in res["result"]["data_array"]:
    print(row)

In [0]:
from pyspark.sql import functions as F
display(spark.table("workspace.patient_notes.silver_chunks")
        .filter(F.lower("chunk_text").rlike("s/p|status post|operated for"))
        .select("patient_id", "note_id", "chunk_text"))